# Manage


In [ ]:
# ---------------------------------
# Databricks API URL and Tokens
# ---------------------------------
# Python cell in the same workspace notebook
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()

api_url  = ctx.apiUrl().getOrElse(None)     # e.g. https://adb-...azuredatabricks.net
api_token = ctx.apiToken().getOrElse(None)  # personal access token for this session

print(api_url)
print(api_token)  # handle securely, do not log in real code



# ---------------------------------
# Scope Creation
# ---------------------------------
import requests
import json

scope_name = "semis2_scope"  # Scope to be created


DATABRICKS_INSTANCE = api_url  
DATABRICKS_TOKEN = api_token  
backend_type = "DATABRICKS"     

url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/scopes/create"

# headers = {
#     "Authorization": f"Bearer {DATABRICKS_TOKEN}",
#     "Content-Type": "application/json"
# }

# payload = {
#     "scope": scope_name
# }

# response = requests.post(url, headers=headers, data=json.dumps(payload))

# if response.status_code == 200:
#     print(f"Secret scope '{scope_name}' created successfully.")
# else:
#     print("Failed to create secret scope.")
#     print("Status Code:", response.status_code)
#     print("Response:", response.text)

# ---------------------------------
# Secret Creation
# ---------------------------------
import requests
import json

scope = "semis2_scope"           # Already existing scope
secret_key = "DB_NAME"        # Name of the secret entry
# secret_value = "https://j5pclnsc-80.use.devtunnels.ms/" # Value to store securely
dbutils.widgets.text("secret_value", "", "Google Drive Refresh Token:")
secret_value = dbutils.widgets.get("secret_value")

url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/put"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "scope": scope,
    "key": secret_key,
    "string_value": secret_value
}

response = requests.post(url, headers=headers, data=json.dumps(payload))

if response.status_code == 200:
    print(f"Secret '{secret_key}' created successfully in scope '{scope}'.")
else:
    print("Failed to create secret.")
    print("Status:", response.status_code)
    print("Response:", response.text)



# ---------------------------------
# Delete Scope and All keys
# ---------------------------------
import requests
scope = "my_secret_scope_2"  # Scope you want to delete

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}"
}

list_url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/list?scope={scope}"
resp = requests.get(list_url, headers=headers)

if resp.status_code == 400:
    print(f"Scope '{scope}' does not exist.")
    exit()

secrets = resp.json().get("secrets", [])

for s in secrets:
    key = s["key"]
    del_url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/delete"
    requests.post(del_url, headers=headers, json={"scope": scope, "key": key})
    print(f"Deleted secret: {key}")

del_scope_url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/scopes/delete"
resp = requests.post(del_scope_url, headers=headers, json={"scope": scope})

if resp.status_code == 200:
    print(f"Scope '{scope}' deleted successfully.")
else:
    print("Failed to delete scope:", resp.text)
    
    

# ---------------------------------
# Share Secrets
# ---------------------------------    
import requests
import json

scope = "semis2_scope"
target_principal = "teamatetoshare@gmail.com"
permission_level = "READ"


url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/acls/put"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "scope": scope,
    "principal": target_principal,
    "permission": permission_level
}

response = requests.post(url, headers=headers, data=json.dumps(payload))

if response.status_code == 200:
    print(f"Success: Granted '{permission_level}' access to '{target_principal}' on scope '{scope}'.")
else:
    print("Failed to grant permissions.")
    print("Status:", response.status_code)
    print("Response:", response.text)
